# Khan Academy → Georgian subtitles

Paste YouTube links, get `.srt` subtitle files in Google Drive (**My Drive / ka-subs / subtitles**).

**First time only:**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Click the **key icon** on the left (Secrets) → **Add new secret** → name `GEMINI_API_KEY`, value = your key → switch **Notebook access** on.

**Every time:** run cell 1, then fill in cell 2 and run it. Cell 1 asks once to connect your Google Drive: allow it, so the work is saved there.

In [ ]:
#@title 1. Setup (about 2 minutes) { display-mode: "form" }
REPO = "https://github.com/razmikb/ka-subs-creator"  #@param {type:"string"}
import os, sys, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE. Menu Runtime → Change runtime type → T4 GPU (it still works without, just slowly).")
from google.colab import drive
drive.mount("/content/drive")
if os.path.isdir("/content/ka-subs-creator/.git"):
    !git -C /content/ka-subs-creator pull -q
else:
    !git clone -q {REPO} /content/ka-subs-creator
!pip install -q -U "yt-dlp[default]" google-genai git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git
sys.path.insert(0, "/content/ka-subs-creator")
print("Ready.")

In [ ]:
#@title 2. Make subtitles { display-mode: "form" }
#@markdown Links: one or more YouTube video links, a playlist, or a channel, separated by spaces.
LINKS = ""  #@param {type:"string"}
#@markdown Gemini model (change only if Gemini says it doesn't exist any more).
MODEL = "gemini-3.8-flash"  #@param {type:"string"}
from pathlib import Path
from google.colab import userdata
from ka_subs.pipeline import run_batch
try:
    KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    KEY = None
    print("No GEMINI_API_KEY in Secrets, so the words will be done by hand in AI Studio.\n")
run_batch(LINKS, Path("/content/drive/MyDrive/ka-subs"), KEY, MODEL)

### If a check found skipped speech
Listen at the link the check gave. If speech really is missing from the subtitles, put that
video's link and the stretch of time below (like `1:20` to `2:05`) and run the cell.
It writes the missing words, puts them in the right place and makes the subtitles again.

In [ ]:
#@title 3. Fill a gap { display-mode: "form" }
LINK = ""  #@param {type:"string"}
FROM = "0:00"  #@param {type:"string"}
TO = "0:00"  #@param {type:"string"}
from pathlib import Path
from google.colab import userdata
from ka_subs.common import Stop, from_clock
from ka_subs.gaps import fill_gap
from ka_subs.gemini import DEFAULT_MODEL
try:
    KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    KEY = None
try:
    fill_gap(LINK.strip(), from_clock(FROM), from_clock(TO),
             Path("/content/drive/MyDrive/ka-subs"), KEY, globals().get("MODEL", DEFAULT_MODEL))
except Stop as exc:
    print(f"STOPPED: {exc}")

### What the messages mean
- **Skipped**: the video already has Georgian subtitles.
- **Waiting for you**: Gemini's free daily limit ran out (or there's no key). Open the video's folder in Drive and follow `NEXT_STEPS.txt`, then run cell 2 again. Finished steps are not repeated.
- **Gap of N s**: no subtitle for a while. Usually the narrator is drawing quietly. Listen at the link to be sure.
- **Stretched over N s**: one subtitle stays on screen too long. This often means some speech is missing from the text. Listen at the link.
- **The end may be missing**: the subtitles stop well before the video ends.
- **LOW words/min**: Gemini may have shortened that part. Delete its `partN.txt` and the video's `<id>.txt` in Drive and run again.
- **YouTube refused**: YouTube blocked Google's servers. The message says how to download the audio on your laptop instead.